# Parallel GPU Computing (PGC) · Lab Evaluation 2
## Theme 7: CUDA Warp Divergence Microarchitectural Analysis

**Team Details:**
- **Adarsh Walad** (`01fe24bci`) — Experimental Design & Presentation Slides
- **Nilaksha Nath** (`01fe24bci087`) — Benchmarking Scripts & Statistical Analysis
- **Bachu venkata siva jaideep** (`01fe24bci084`) — GPU Architecture Design & CUDA Compilation
- **Prathamesh Eedigi** (`01fe24bci006`) — SIMT Active Mask Diagrams & Documentation

---
### Abstract & Research Aim
Evaluate Single Instruction, Multiple Threads (SIMT) branch divergence in CUDA on the NVIDIA Ada Lovelace (SM 8.9) microarchitecture. Compare four execution strategies across 16.78M single-precision floating point elements: (1) Divergent Branching, (2) Warp-Aligned Branching, (3) Data-Dependent Branching, and (4) Branch-Free Predication.

### Step 1: Hardware & Toolchain Verification
Inspect the NVIDIA GPU, driver version, compute capability, and CUDA compiler driver (`nvcc`).

In [ ]:
!nvidia-smi
!nvcc --version

### Step 2: Inspect CUDA Source Implementation
Verify the four kernel implementations inside `../src/warp_divergence.cu`:
- **Divergent Kernel**: Alternates branch choice per thread (`i % 2 == 0`), producing 50% active lane masking within every warp.
- **Warp-Aligned Kernel**: Evaluates branch choice at 32-thread boundaries (`((i / 32) % 2) == 0`), guaranteeing 100% active lanes across all warps.
- **Data-Dependent Kernel**: Branches on runtime pseudo-random memory values (`input[i] > threshold`), simulating real-world unstructured workloads.
- **Branch-Free Kernel**: Predicated execution evaluating both candidate paths unconditionally and selecting via ternary operator (`selp`).

In [ ]:
with open('../src/warp_divergence.cu', 'r') as f:
    cuda_code = f.read()
print(f"CUDA source loaded successfully ({len(cuda_code.splitlines())} lines).")

### Step 3: Compilation and Execution Command
Targeting NVIDIA RTX 4500 Ada Generation with `sm_89`:

In [ ]:
# Windows MSVC / NVCC Command
# nvcc -O2 -std=c++14 -arch=sm_89 ../src/warp_divergence.cu -o ../bin/warp_divergence.exe
# ../bin/warp_divergence.exe

# Linux / Unix Command
# nvcc -O2 -std=c++14 -arch=sm_89 ../src/warp_divergence.cu -o ../bin/warp_divergence
# ../bin/warp_divergence

### Step 4: Performance Analysis & Visualization
Load benchmark metrics from `timing_data.csv` and visualize execution latency, throughput, speedup ($S = T_{\text{baseline}} / T_{\text{variant}}$), and execution-time ratio ($R = T_{\text{variant}} / T_{\text{baseline}}$).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../results/timing_data.csv')
display(df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ['#E74C3C', '#2ECC71', '#3498DB', '#9B59B6']

# Panel 1: Execution Time
ax1.bar(df['kernel_name'], df['avg_time_ms'], color=colors, edgecolor='#1A252F')
ax1.set_title('Kernel Latency (ms) [Lower is Better]', fontweight='bold')
ax1.set_ylabel('Execution Time (ms)')
ax1.set_xticklabels(df['kernel_name'], rotation=20, ha='right')
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Panel 2: Speedup vs Baseline
ax2.bar(df['kernel_name'], df['speedup_vs_baseline'], color=colors, edgecolor='#1A252F')
ax2.axhline(1.0, color='red', linestyle='--', label='1.0x Baseline')
ax2.set_title('Speedup S = T_base / T_var [> 1.0 is Faster]', fontweight='bold')
ax2.set_ylabel('Speedup Factor')
ax2.set_xticklabels(df['kernel_name'], rotation=20, ha='right')
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.legend()

plt.tight_layout()
plt.show()